To run this, press "*Runtime*" and press "*Run all*" on your A100 Google Colab Pro instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.7.1" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Mamba is supported only on torch==2.7.1. If you have newer torch versions, please wait 30 minutes!
!uv pip install --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Nemotron-3-Nano-30B-A3B",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    unsloth_force_compile = True,
    attn_implementation = "eager",
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


A new version of the following files was downloaded from https://huggingface.co/unsloth/Nemotron-3-Nano-30B-A3B:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Nemotron_H patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


modeling_nemotron_h.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/unsloth/Nemotron-3-Nano-30B-A3B:
- modeling_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Nemotron-3-Nano-30B-A3B does not have a padding token! Will use pad_token = <SPECIAL_999>.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [3]:
RANK_LoRA = 32 # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
model = FastLanguageModel.get_peft_model(
    model,
    r = RANK_LoRA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "in_proj", "out_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 29102005,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'in_proj', 'out_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Nemotron` format for conversation style finetunes. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy. Nemotron renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

#### **Competition dataset**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/nvidia-nemotron-model-reasoning-challenge.zip'
extract_dir = '/content/Data'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted '{zip_file_path}' to '{extract_dir}'")
print("Contents of extracted directory:")
print(os.listdir(extract_dir))

Extracted '/content/drive/MyDrive/nvidia-nemotron-model-reasoning-challenge.zip' to '/content/Dataset'
Contents of extracted directory:
['test.csv', 'train.csv']


In [6]:
import pandas as pd
df = pd.read_csv("/content/Dataset/train.csv")

In [8]:
def formatting_prompts(row):
    instruction = (
        "You are an elite logical reasoning expert. Your task is to decode hidden transformation rules from examples and apply them to a target query.\n"
        "Before giving the final answer, you MUST think step-by-step inside <think>...</think> tags.\n"
        "After your <think> process, output ONLY the final answer enclosed strictly within \\boxed{}."
    )
    question = str(row['prompt']).strip()

    # CHỈ CÓ SYSTEM VÀ USER. KHÔNG CÓ ASSISTANT.
    prompt_text = (
        f"<extra_id_0>System\n{instruction}\n"
        f"<extra_id_1>User\n{question}\n"
        f"<extra_id_1>Assistant\n"
    )

    # GRPOTrainer yêu cầu cột đầu vào bắt buộc tên là 'prompt'
    # Bạn cũng cần giữ lại cột 'answer' gốc để làm đáp án đối chiếu khi chấm điểm
    return {
        "prompt": prompt_text,
        "ground_truth": str(row['answer']).strip()
    }

# Áp dụng
from datasets import Dataset
rl_dataset = Dataset.from_pandas(df)
rl_dataset = rl_dataset.map(
    formatting_prompts
    , remove_columns=list(df.columns)
)

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

In [9]:
rl_dataset

Dataset({
    features: ['prompt', 'ground_truth'],
    num_rows: 9500
})

In [14]:
import re

# 1. Hàm chấm điểm Format (Khuyến khích model giữ đúng cấu trúc)
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    print(f"\n[DEBUG]\n{completions[0][0]['content']}\n")
    for completion in completions:
        # Nếu có đủ cả thẻ <think> và \boxed{}, thưởng 0.5 điểm
        if "<think>" in completion and "</think>" in completion and "\\boxed{" in completion:
            rewards.append(0.5)
        else:
            rewards.append(-10) # Không đúng format thì 0 điểm
    return rewards

# 2. Hàm chấm điểm Chính xác (Quan trọng nhất)
def correctness_reward_func(prompts, completions, ground_truth, **kwargs):
    rewards = []
    # completions là danh sách các câu trả lời model tự sinh ra
    # ground_truth là đáp án đúng từ dataset
    print(f"\n[DEBUG]\n{completions[0][0]['content']}\n")
    for completion, truth in zip(completions, ground_truth):
        # Trích xuất đáp án trong \boxed{}
        match = re.search(r'\\boxed\{(.+?)\}', completion)
        if match:
            pred = match.group(1).strip()
            if pred == truth:
                rewards.append(2.0) # Đúng tuyệt đối -> Thưởng lớn!
            else:
                rewards.append(-10) # Giải sai -> Phạt nhẹ để nó rút kinh nghiệm
        else:
            rewards.append(-50) # Không thèm ra đáp án -> Phạt nặng!
    return rewards

In [20]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [21]:
from trl import GRPOConfig, GRPOTrainer
import torch

torch.cuda.empty_cache()

# Cấu hình GRPO (Rất nhạy cảm với VRAM)
grpo_config = GRPOConfig(
    # output_dir="/kaggle/working/nemotron-grpo",
    learning_rate=1e-5, # RL cần LR nhỏ hơn SFT rất nhiều (thường 1e-5 hoặc 5e-6)

    # THUẬT TOÁN GRPO CỐT LÕI
    num_generations=1, # Model sẽ đẻ ra 4 cách giải khác nhau cho 1 bài toán để tự so sánh (Cẩn thận: càng cao càng tốn RAM)
    max_completion_length=256, # Giới hạn độ dài câu trả lời để tránh OOM

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=50,
    logging_steps=1,

    bf16=True,
    optim="paged_adamw_8bit", # Bắt buộc dùng 8-bit để sinh tồn trên Kaggle
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_vllm=False,
)

# Khởi tạo Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, correctness_reward_func], # Đưa hàm chấm điểm vào đây
    args=grpo_config,
    train_dataset=rl_dataset, # Data chỉ có 'prompt' và 'ground_truth'
    processing_class=tokenizer,
)

print("\n🚀 Bắt đầu quá trình Reinforcement Learning (GRPO)...")
trainer.train()

ValueError: GRPO requires at least 2 generations per prompt to calculate the advantages. You provided 1, which is less than the minimum required.

#### **OpenMathReasoning-mini**

In [ ]:
from datasets import load_dataset
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

We now convert the reasoning dataset into conversational format:

In [ ]:
def generate_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

dataset = dataset.map(generate_conversation, batched = True)

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

We now have to apply the chat template for `Nemotron` onto the conversations, and save it to `text`.

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

Let's see how the chat template did!

In [ ]:
dataset[100]['text']

'<|im_start|>system\n<|im_end|>\n<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? Are they overlapping? If the distance between centers is such that when the hands are pointing towards each other, the tips are closest, and when pointing away, they\'re farthest. But maybe the clocks are placed at a certain distance apart so that the hands\' tips have varying distances based on their positions.\n\nLet me think. Let\'s model each clock as a circle with radius r. The centers of the two clocks are separated by a distance d, which we need to find. The problem states that the minimum and maximum distances between the edges (tips) of their hands are m and M, respectively. So, the hands are moving, but since they have the same speed, maybe their angles relative to each other are fixed? Or do the hands move independently?\n\nWait, the problem says they "may not show the same hour," which probably means that their hands can be at any angle relative to each other. But since both clocks are operating at the same speed, if they start at different times, their hands will maintain a constant angle difference. Wait, but the minute and hour hands move at different speeds. Wait, the problem says "the same speed," but in reality, the hour and minute hands move at different speeds. Hmm, maybe in this problem, each clock is simplified such that there\'s a single hand moving at a constant speed? Wait, but in standard clocks, the hour and minute hands have different speeds. Maybe the problem is referring to each clock having just one hand? The problem says "hands," plural, so perhaps both clocks have hour and minute hands, but they might not show the same time. But the problem mentions the distance between the edges of their hands. Wait, maybe it\'s the tips of the hands? So perhaps each clock has an hour hand and a minute hand, but the problem is considering the distance between the tips of the hands from one clock to the other?\n\nWait, the problem says "the edges of their hands" – maybe the hands refer to the clock hands, so each clock has hands (like hour, minute, etc.), but the problem is talking about the distance between the edges (tips) of the hands from each clock. Wait, but if each clock has multiple hands, then the distance between edges would vary depending on which hands you\'re comparing. Hmm, maybe the problem is simplified such that each clock has only one hand? Or perhaps the problem is referring to the distance between the tips of the corresponding hands? For example, the tip of the hour hand of one clock to the tip of the hour hand of the other, and similarly for the minute hand? But the problem states "the edges of their hands," so maybe considering all possible pairs of hands between the two clocks? But that seems complicated. Wait, maybe the pr

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|im_start|>system\n<|im_end|>\n<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? Are they overlapping? If the distance between centers is such that when the hands are pointing towards each other, the tips are closest, and when pointing away, they\'re farthest. But maybe the clocks are placed at a certain distance apart so that the hands\' tips have varying distances based on their positions.\n\nLet me think. Let\'s model each clock as a circle with radius r. The centers of the two clocks are separated by a distance d, which we need to find. The problem states that the minimum and maximum distances between the edges (tips) of their hands are m and M, respectively. So, the hands are moving, but since they have the same speed, maybe their angles relative to each other are fixed? Or do the hands move independently?\n\nWait, the problem says they "may not show the same hour," which probably means that their hands can be at any angle relative to each other. But since both clocks are operating at the same speed, if they start at different times, their hands will maintain a constant angle difference. Wait, but the minute and hour hands move at different speeds. Wait, the problem says "the same speed," but in reality, the hour and minute hands move at different speeds. Hmm, maybe in this problem, each clock is simplified such that there\'s a single hand moving at a constant speed? Wait, but in standard clocks, the hour and minute hands have different speeds. Maybe the problem is referring to each clock having just one hand? The problem says "hands," plural, so perhaps both clocks have hour and minute hands, but they might not show the same time. But the problem mentions the distance between the edges of their hands. Wait, maybe it\'s the tips of the hands? So perhaps each clock has an hour hand and a minute hand, but the problem is considering the distance between the tips of the hands from one clock to the other?\n\nWait, the problem says "the edges of their hands" – maybe the hands refer to the clock hands, so each clock has hands (like hour, minute, etc.), but the problem is talking about the distance between the edges (tips) of the hands from each clock. Wait, but if each clock has multiple hands, then the distance between edges would vary depending on which hands you\'re comparing. Hmm, maybe the problem is simplified such that each clock has only one hand? Or perhaps the problem is referring to the distance between the tips of the corresponding hands? For example, the tip of the hour hand of one clock to the tip of the hour hand of the other, and similarly for the minute hand? But the problem states "the edges of their hands," so maybe considering all possible pairs of hands between the two clocks? But that seems complicated. Wait, maybe the pr

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                          <think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? Are they overlapping? If the distance between centers is such that when the hands are pointing towards each other, the tips are closest, and when pointing away, they\'re farthest. But maybe the clocks are placed at a certain distance apart so that the hands\' tips have varying distances based on their positions.\n\nLet me think. Let\'s model each clock as a circle with radius r. The centers of the two clocks are separated by a distance d, which we need to find. The problem states that the minimum and maximum distances between the edges (tips) of their hands are m and M, respectively. So, the hands are moving, but since they have the same speed, maybe their angles relative to each other are fixed? Or do the hands move independently?\n\nWait, the problem says they "may not show the same hour," which probably means that their hands can be at any angle relative to each other. But since both clocks are operating at the same speed, if they start at different times, their hands will maintain a constant angle difference. Wait, but the minute and hour hands move at different speeds. Wait, the problem says "the same speed," but in reality, the hour and minute hands move at different speeds. Hmm, maybe in this problem, each clock is simplified such that there\'s a single hand moving at a constant speed? Wait, but in standard clocks, the hour and minute hands have different speeds. Maybe the problem is referring to each clock having just one hand? The problem says "hands," plural, so perhaps both clocks have hour and minute hands, but they might not show the same time. But the problem mentions the distance between the edges of their hands. Wait, maybe it\'s the tips of the hands? So perhaps each clock has an hour hand and a minute hand, but the problem is considering the distance between the tips of the hands from one clock to the other?\n\nWait, the problem says "the edges of their hands" – maybe the hands refer to the clock hands, so each clock has hands (like hour, minute, etc.), but the problem is talking about the distance between the edges (tips) of the hands from each clock. Wait, but if each clock has multiple hands, then the distance between edges would vary depending on which hands you\'re comparing. Hmm, maybe the problem is simplified such that each clock has only one hand? Or perhaps the problem is referring to the distance between the tips of the corresponding hands? For example, the tip of the hour hand of one clock to the tip of the hour hand of the other, and similarly for the minute hand? But the problem states "the edges of their hands," so maybe considering all possible pairs of hands between the two clocks? But that seems complicated. Wait, maybe the problem is just considering the distance between the tips of the hands of the same type? For example, the hour hands of each clock? Or maybe it\'s considering the distance between any two hands from different clocks? The problem is a bit ambiguous here.\n\nWait, let me read the pr

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
59.752 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,252 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 220,968,448 of 31,798,905,792 (0.69% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.891400
2,0.799300
3,0.913400
4,0.902700
5,0.896800
6,0.853900
7,0.695000
8,0.729100
9,0.845000
10,0.789000


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1615.0025 seconds used for training.
26.92 minutes used for training.
Peak reserved memory = 76.904 GB.
Peak reserved memory for training = 17.152 GB.
Peak reserved memory % of max memory = 96.957 %.
Peak reserved memory for training % of max memory = 21.624 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference!

In [ ]:
messages = [
    {"role" : "user", "content" : "Continue the sequence: 1, 1, 2, 3, 5, 8,"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20,
    use_cache = False,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Okay, so I need to figure out the next number in the sequence: 1, 1, 2, 3, 5, 8. Hmm, let's see. I remember something about Fibonacci numbers from math class. Let me think. The Fibonacci sequence starts with 1, 1, and then each subsequent number is the sum of the two preceding ones. Let me check if that's the case here.

Starting with the first two numbers: 1 and 1. The next number should be 1 + 1, which is 2. That matches the third number in the sequence. Then the fourth number would be 1 + 2, which is 3. Yep, that's right. The fifth number is 2 + 3, which equals 5. Perfect, that's the fifth term. The sixth number is 3 + 5, which is 8. So the sequence given is indeed the Fibonacci sequence.

Now, following this pattern, the next number after 8 should be the sum of the last two numbers, which are 5 and 8. So 5 + 8 equals 13. Therefore, the next number in the sequence should be 13. Let me just double-check to make sure I didn't make a mistake. 5 plus 8 is definitely 13. Yep, that seems right. I don't think there's any other pattern here. It's straightforward Fibonacci. So the answer should be 13.
</think>To continue the sequence \(1, 1, 2, 3, 5, 8\), we observe that each term after the first two is the sum of the two preceding terms. This is the Fibonacci sequence.

1. The first term is \(1\).
2. The second term is \(1\).
3. The third term is \(1 + 1 = 2\).
4. The fourth term is \(1 + 2 = 3\).
5. The fifth term is \(2 + 3 = 5\).
6. The sixth term is \(3 + 5 = 8\).

Following this pattern, the next term (the seventh term) is the sum of the fifth and sixth terms:
\[
5 + 8 = 13
\]

Thus, the next number in the sequence is \(\boxed{13}\).<|im_end|>

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("nemotron_lora")  # Local saving
tokenizer.save_pretrained("nemotron_lora")
# model.push_to_hub("your_name/nemotron_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/nemotron_lora", token = "YOUR_HF_TOKEN") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "nemotron_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("nemotron_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/nemotron_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("nemotron_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/nemotron_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("nemotron_lora")
    tokenizer.save_pretrained("nemotron_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/nemotron_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/nemotron_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("nemotron_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/nemotron_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("nemotron_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/nemotron_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("nemotron_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/nemotron_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/nemotron_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `nemotron_finetune.Q8_0.gguf` file or `nemotron_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
</div>